# 🩺 Mini-ChatGPT na nazwach chorób
### Sympozjum „Nowoczesne technologie w pediatrii" · Kamil Jędryczek

Ten notebook buduje **mały model językowy** — dokładnie ten sam mechanizm, na którym opiera się ChatGPT, tylko ok. 10 milionów razy mniejszy (ok. 80 tys. parametrów; modele klasy GPT-4 mają ich szacunkowo ~10¹²). Zamiast przewidywać następne **słowo** w zdaniu, przewidujemy następną **literę** w nazwie choroby.

**Jak uruchomić:** menu `Środowisko wykonawcze` → `Uruchom wszystko` (albo Ctrl+F9). Działa też na telefonie.

Trening trwa kilka minut. W tym czasie można czytać komentarze 🙂

In [ ]:
# ── KONFIGURACJA ──────────────────────────────────────────────
# Który zbiór nazw trenujemy? Do wyboru: "choroby", "leki", "dinozaury"
ZBIOR = "choroby"

REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data"

# Skąd wziąć dane (jedna nazwa na linię, bez nagłówka)
DANE = {
    "choroby":   f"{REPO}/choroby.csv",   # łacińskie nazwy chorób (Wikipedia PL/LA, CC BY-SA)
    "leki":      f"{REPO}/leki.csv",      # nazwy handlowe leków (Rejestr Produktów Leczniczych)
    "dinozaury": "https://raw.githubusercontent.com/junosuarez/dinosaurs/master/dinosaurs.csv",
}

# Nastawy dobrane automatyczną optymalizacją (eksperymenty/AUTOOPTYMALIZACJA.md):
# 12 epok z szybszym uczeniem = ok. 15 s treningu, ~98% nazw nowych,
# nazwy brzmią jak prawdziwe łacińskie. NIE trenuj dłużej — model zacznie zapamiętywać.
LICZBA_EPOK = 12
TEMPO_UCZENIA = 0.003   # learning rate: 3× wyższy niż domyślny → model uczy się szybciej
TEMPERATURA = 0.4   # niższa = ostrożniejsze losowanie = nazwy bardziej „łacińskie"

# Aplikacja dla sali (ostatnia komórka) korzysta z Gradio. Instalujemy je TERAZ,
# przed treningiem — w Colabie trwa to kilkanaście sekund i nie blokuje pokazu na końcu.
try:
    import gradio  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio"], check=True)

In [ ]:
# ── DANE ──────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd

# Jeśli plik leży lokalnie (klon repo) — bierzemy go; inaczej pobieramy z GitHuba.
# Dzięki temu notebook działa też bez internetu, gdy padnie Wi-Fi na sali.
zrodlo = DANE[ZBIOR]
plik = os.path.basename(zrodlo)
for kandydat in (plik, f"data/{plik}", f"../data/{plik}"):
    if os.path.exists(kandydat):
        zrodlo = kandydat
        break

df = pd.read_csv(zrodlo, header=None, names=["nazwa"], sep="\t")
nazwy = (
    df["nazwa"].dropna().astype(str).str.strip().str.lower()
      .drop_duplicates().tolist()
)

# Tasujemy kolejność nazw (ze stałym ziarnem 42, więc każde uruchomienie daje tę samą kolejność).
# Po co? Plik jest ułożony prawie alfabetycznie, a podczas treningu odkładamy na bok
# OSTATNIE 10% nazw jako „sprawdzian". Bez tasowania sprawdzian składałby się z samych
# nazw na „t", „u", „v", „x", „z" — jak egzamin wyłącznie z ostatniego rozdziału podręcznika.
rng = np.random.default_rng(42)
nazwy = [nazwy[i] for i in rng.permutation(len(nazwy))]
print(f"Wczytano {len(nazwy)} nazw z: {zrodlo}\nPrzykłady:")
for n in nazwy[:8]:
    print("  •", n)

## Tokeny specjalne

Każdą nazwę opakowujemy w dwa znaki specjalne:
- `%` — **START**: „zacznij generować"
- `!` — **STOP**: „to już koniec nazwy"

Czyli `pneumonia` staje się `%pneumonia!`. Bez `%` model nie wiedziałby, od czego zacząć; bez `!` generowałby w nieskończoność.

In [ ]:
# ── PRZYGOTOWANIE DANYCH ──────────────────────────────────────
import numpy as np

sekwencje = ["%" + n + "!" for n in nazwy]

# Słownik: każdy znak dostaje numer (0 rezerwujemy na "puste miejsce" / padding)
znaki = sorted(set("".join(sekwencje)))
znak2id = {z: i + 1 for i, z in enumerate(znaki)}
id2znak = {i: z for z, i in znak2id.items()}
WIELKOSC_SLOWNIKA = len(znaki) + 1
print(f"Słownik ma {WIELKOSC_SLOWNIKA} tokenów: {''.join(znaki)}")

# X = nazwa bez ostatniego znaku, Y = nazwa bez pierwszego znaku.
# Model uczy się: po każdym prefiksie przewidź NASTĘPNY znak.
#   X: %pneumonia     Y: pneumonia!
maks_dl = max(len(s) for s in sekwencje)
X = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
Y = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
for i, s in enumerate(sekwencje):
    ids = [znak2id[z] for z in s]
    X[i, :len(ids) - 1] = ids[:-1]
    Y[i, :len(ids) - 1] = ids[1:]
print(f"Macierz treningowa: {X.shape} (nazwy × pozycje znaków)")

## Model

Trzy warstwy, które robią całą magię:
1. **Embedding** — każda litera dostaje swój wektor liczb („znaczenie" litery),
2. **GRU** — sieć rekurencyjna z pamięcią: czyta literę po literze i pamięta kontekst,
3. **Dense + softmax** — zamienia pamięć na **rozkład prawdopodobieństwa** następnej litery.

ChatGPT różni się w środku (Transformer zamiast GRU) i skalą — ale wejście i wyjście ma identyczne: *kontekst → prawdopodobieństwa następnego tokenu*.

> **Dlaczego tylko 12 epok?** Przy dłuższym treningu model przestaje uczyć się *reguł* budowy łacińskich nazw, a zaczyna odtwarzać z pamięci konkretne nazwy ze zbioru. Sprawdziliśmy to pomiarem (katalog `eksperymenty/`): przy 80 epokach tylko **10%** wygenerowanych nazw było nowych. Automatyczna optymalizacja (ponad 400 pomiarów + ślepa ocena nazw) pokazała, że 12 epok z wyższym tempem uczenia daje nazwy co najmniej tak dobre jak 30 epok — w **ok. 2,5× krótszym czasie**, co na żywo ma znaczenie.


In [ ]:
# ── TRENING ───────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Embedding(WIELKOSC_SLOWNIKA, 64, mask_zero=True),
    keras.layers.GRU(128, return_sequences=True, dropout=0.2),   # dropout = mniej zapamiętywania
    keras.layers.Dense(WIELKOSC_SLOWNIKA, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=TEMPO_UCZENIA))

# 10% nazw (ostatnie po przetasowaniu, czyli losowe) odkładamy na bok — model ich nie widzi w treningu.
# Dzięki temu widać, czy naprawdę się UCZY, czy tylko ZAPAMIĘTUJE.
historia = model.fit(X, Y, epochs=LICZBA_EPOK, validation_split=0.1, verbose=2)

In [ ]:
# ── KRZYWA UCZENIA ────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"], label="trening (nazwy, które model widzi)")
plt.plot(historia.history["val_loss"], label="walidacja (nazwy odłożone na bok)")
plt.xlabel("Epoka"); plt.ylabel("Błąd (loss)")
plt.title("Dopóki walidacja spada — model uczy się reguł; gdy rośnie — zapamiętuje")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Generowanie — serce autoregresji

Pętla jest banalna:
1. pokaż modelowi dotychczasowy kontekst (zaczynamy od `%`),
2. model zwraca **rozkład prawdopodobieństwa** wszystkich możliwych następnych liter,
3. **wylosuj** literę z tego rozkładu (temperatura steruje „odwagą" losowania),
4. dopisz literę do kontekstu i wróć do punktu 1 — aż wypadnie `!`.

Model **nigdy nie mówi „nie wiem"** — zawsze zwraca jakiś rozkład i zawsze coś wylosujemy. Zapamiętajcie to zdanie przy temacie halucynacji.

Na koniec sprawdzamy rzecz kluczową: **czy ta nazwa w ogóle istnieje w zbiorze treningowym?** Jeśli nie — model niczego nie skopiował, tylko nauczył się reguł budowy łacińskich nazw i złożył z nich nową.


In [ ]:
# ── GENEROWANIE ───────────────────────────────────────────────
ZNANE = set(nazwy)   # zbiór treningowy — do sprawdzania, czy nazwa jest NAPRAWDĘ nowa

import matplotlib.pyplot as plt

def generuj(temperatura=TEMPERATURA, maks_znakow=40):
    """Zwraca (nazwa, kroki). Każdy krok = pełny rozkład P(następna litera | kontekst)."""
    kontekst, kroki = "%", []
    for _ in range(maks_znakow):
        ids = np.array([[znak2id[z] for z in kontekst]])
        rozklad = model(ids, training=False).numpy()[0, -1].astype("float64")  # P(następna litera | kontekst)
        rozklad = np.log(rozklad + 1e-9) / temperatura           # temperatura
        rozklad = np.exp(rozklad) / np.exp(rozklad).sum()
        rozklad[0] = 0; rozklad /= rozklad.sum()                 # nie losujemy paddingu
        wybor = int(np.random.choice(len(rozklad), p=rozklad))
        kroki.append({"kontekst": kontekst, "rozklad": rozklad, "wybor": wybor})
        if id2znak[wybor] == "!":
            break
        kontekst += id2znak[wybor]
    return kontekst[1:], kroki   # bez tokenu startu


def werdykt(nazwa):
    """Czy model coś wymyślił, czy tylko odtworzył nazwę ze zbioru treningowego?"""
    if nazwa in ZNANE:
        return f"⚠️  \"{nazwa}\" JUŻ ISTNIEJE w zbiorze treningowym — model ją odtworzył, nie wymyślił."
    return f"✨ \"{nazwa.upper()}\" — tej nazwy NIE MA w zbiorze treningowym. Model ją wymyślił."


def pokaz_kroki(kroki):
    """Tekstowy ślad generowania: kontekst → top 5 kandydatów → wylosowana litera."""
    for krok in kroki:
        p, w = krok["rozklad"], krok["wybor"]
        top5 = np.argsort(p)[::-1][:5]
        opis = "  ".join(f"{id2znak[i]!r}:{p[i]:.0%}" for i in top5)
        print(f"  kontekst: {krok['kontekst']:<28} → wylosowano {id2znak[w]!r:<5} "
              f"(p={p[w]:.0%})   top5: {opis}")


def histogramy(kroki, kolumny=4):
    """Rozkład prawdopodobieństwa w każdym kroku. Pomarańczowy słupek = litera wylosowana."""
    litery = [id2znak[i] for i in range(1, WIELKOSC_SLOWNIKA)]
    wiersze = int(np.ceil(len(kroki) / kolumny))
    fig, axes = plt.subplots(wiersze, kolumny, figsize=(4.6 * kolumny, 2.9 * wiersze),
                             squeeze=False)
    for ax, krok in zip(axes.ravel(), kroki):
        p, w = krok["rozklad"][1:], krok["wybor"]
        kolory = ["#3A1C7D"] * len(p)
        kolory[w - 1] = "#FFAF7B"                     # wylosowana litera
        ax.bar(litery, p, color=kolory)
        ax.set_ylim(0, 1)
        ax.set_title(f"{krok['kontekst']}  →  {id2znak[w]!r}", fontsize=11)
        ax.tick_params(axis="x", labelsize=7)
        ax.grid(axis="y", alpha=0.25)
    for ax in axes.ravel()[len(kroki):]:
        ax.axis("off")
    fig.suptitle("P(następna litera | kontekst) — rozkład ostrzeje, gdy kontekst rośnie", y=1.0)
    fig.tight_layout()
    plt.show()


nazwa, kroki = generuj()
print("Ślad generowania — krok po kroku:\n")
pokaz_kroki(kroki)
print("\n" + werdykt(nazwa))
histogramy(kroki)

In [ ]:
# ── 10 NOWYCH NAZW ────────────────────────────────────────────
nowe = 0
for _ in range(10):
    nazwa, _ = generuj()
    if nazwa in ZNANE:
        print(f" • {nazwa:<40} ⚠️  istnieje w zbiorze treningowym")
    else:
        print(f" • {nazwa:<40} ✨ nowa")
        nowe += 1
print(f"\n{nowe}/10 nazw nie występuje w zbiorze treningowym.")

### Eksport śladu generowania (do slajdu w prezentacji)

Poniższa komórka generuje jedną nazwę i zapisuje wszystkie kroki (rozkłady, wybory) jako JSON w formacie slajdu „Krok po kroku" (`GENERATION_STEPS`; pole `desc` uzupełnia się ręcznie). Uruchamiaj, aż wypadnie nazwa, która Ci się podoba — potem skopiuj JSON.

In [ ]:
# ── ŚLAD GENEROWANIA → JSON POD SLAJD ─────────────────────────
import json

nazwa, kroki = generuj()
print(werdykt(nazwa) + f"   ({len(kroki)} kroków)\n")

def pokaz(z):
    """Spacja na slajdzie byłaby niewidoczna — pokazujemy ją jako ␣."""
    return "␣" if z == " " else z

eksport = []
for krok in kroki:
    p, w = krok["rozklad"], krok["wybor"]
    top5 = np.argsort(p)[::-1][:5]
    probs = [[pokaz(id2znak[int(i)]), round(float(p[i]), 2)] for i in top5]
    if w not in top5:   # wylosowana litera spoza pierwszej piątki — też musi mieć słupek
        probs.append([pokaz(id2znak[w]), round(float(p[w]), 2)])
    probs.append(["…", round(max(0.0, 1 - sum(x for _, x in probs)), 2)])
    eksport.append({
        "context": "".join(pokaz(z) for z in krok["kontekst"]),
        "probs": probs,
        "chosen": pokaz(id2znak[w]),
        "chosen_p": round(float(p[w]), 2),
        "top": pokaz(id2znak[int(top5[0])]),
        "desc": "",   # opis kroku po polsku — do uzupełnienia ręcznie
    })

# Wklej jako GENERATION_STEPS w slides/inference_steps.js
print(json.dumps(eksport, ensure_ascii=False, indent=2))

## 📱 Aplikacja dla całej sali (Gradio)

Ostatnia komórka odpala prostą aplikację webową z **publicznym linkiem** (`*.gradio.live`, ważny 72 h). Wystarczy zrobić z niego kod QR — każdy telefon na sali może generować nazwy, bez logowania.

In [ ]:
# ── GRADIO ────────────────────────────────────────────────────
# (gradio instaluje się w komórce KONFIGURACJA, przed treningiem)
import gradio as gr

def app_generuj(ile, temperatura):
    wyniki = []
    for _ in range(int(ile)):
        nazwa, _ = generuj(temperatura=temperatura)
        znacznik = "  (istnieje w zbiorze)" if nazwa in ZNANE else ""
        wyniki.append(f"• {nazwa}{znacznik}")
    return "\n".join(wyniki)

demo = gr.Interface(
    fn=app_generuj,
    inputs=[
        gr.Slider(1, 10, value=3, step=1, label="Ile nazw wygenerować?"),
        gr.Slider(0.2, 1.5, value=TEMPERATURA, step=0.1, label="Kreatywność (temperatura)"),
    ],
    outputs=gr.Textbox(label="Nowe nazwy — właśnie wymyślone przez model", lines=10),
    title=f"🩺 Generator: {ZBIOR}",
    description="Mały model językowy (GRU) wytrenowany na żywo podczas prelekcji. "
                "Ten sam mechanizm co w ChatGPT — tylko ok. 10 milionów razy mniejszy (ok. 80 tys. parametrów).",
)
demo.queue(default_concurrency_limit=8)   # cała sala naraz: do 8 zapytań równolegle, reszta czeka w kolejce
demo.launch(share=True)